In [1]:
# -*- coding: utf-8 -*-
"""Sprint 3 (Improved): Vision-Language-Assisted Failure Mining

Improvements over baseline Sprint 3:
  - Balanced hard-negative selection (not dominated by one failure category)
  - Confidence-masked pseudo-labels with ignored uncertain pixels
  - Iterative self-training (2 rounds) with staged backbone unfreezing
  - Hard-crop training around boundary/uncertainty regions
  - Grounding DINO as strong teacher for detection pseudo-labels
  - Test-Time Augmentation (TTA) for smoother segmentation predictions
  - Mixed-domain detector fine-tuning with BDD100K source data
"""

# ==============================================================================
# SECTION 1: Setup & Model Loading
# ==============================================================================

import subprocess
import sys

for pkg in [
    "open_clip_torch",
    "transformers",
    "Pillow",
    "tqdm",
    "gdown",
    "opencv-python",
    "kagglehub",
    "python-dotenv",
]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# Grounding DINO -- strong zero-shot detector for pseudo-label generation
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "groundingdino-py",  # PyPI package
])

0

In [2]:
import copy
import glob
import json
import os
import random
import re
import shutil
import zipfile
from collections import defaultdict

import cv2
import gdown
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import ConcatDataset, DataLoader, Dataset, Subset
from torchvision import transforms
from tqdm import tqdm

try:
    from google.colab import userdata
except Exception:
    userdata = None

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# -- Global constants ---------------------------------------------------------

SEED = 17

IMG_SIZE = (512, 1024)  # (H, W) -- matches DeepLabV3+ expected input
IMAGENET_MEAN = [0.485, 0.456, 0.406]  # ImageNet channel means for normalization
IMAGENET_STD = [0.229, 0.224, 0.225]   # ImageNet channel stds for normalization
ROAD_COLOR = (128, 64, 128)  # Cityscapes road class RGB colour
COLOR_TOLERANCE = 5          # Tolerance for matching pseudo-label colours
IGNORE_INDEX = 255           # Label value for ignored (uncertain) pixels

# -- Batch sizes ---------------------------------------------------------------
BATCH_SIZE = 8         # Inference batch size
TRAIN_BATCH_SIZE = 4   # Segmentation training batch size
DET_BATCH_SIZE = 4     # Detection training batch size

# -- Hard negative mining settings --------------------------------------------
HARD_NEG_COUNT = 300          # Number of hard negative frames to mine
HARD_CROPS_PER_FRAME = 2      # Focus crops per frame for boundary training
HARD_CROP_SIZE = (320, 640)   # (H, W) of each focus crop
MAX_FRAMES_PER_CLIP = 18      # Max frames per video clip (diversity limit)

# -- Self-training loop settings ----------------------------------------------
SELF_TRAIN_ROUNDS = 2         # Number of iterative self-training rounds
ROUND_EPOCHS = 10             # Epochs per round
WARMUP_EPOCHS = 2             # Epochs with only head unfrozen
UNFREEZE_LAYER3_EPOCH = 6     # Epoch to start unfreezing backbone layer3

# -- Learning rates (staged unfreezing) ----------------------------------------
HEAD_LR = 5e-4      # Classifier/decoder head
LAYER4_LR = 1e-4     # Backbone layer4 (after warmup)
LAYER3_LR = 5e-5     # Backbone layer3 (late unfreezing)

# -- Pseudo-label quality thresholds ------------------------------------------
PSEUDO_DISAGREE_MARGIN = 0.25   # Minimum margin to keep disagreement pixels
PSEUDO_MIN_VALID_RATIO = 0.55   # Minimum fraction of non-ignored pixels
PSEUDO_MIN_MEAN_WEIGHT = 0.20   # Minimum mean weight for a frame to be trainable
PSEUDO_WEIGHT_FLOOR = 0.05      # Floor weight for confident regions
PSEUDO_PREVIEW_LIMIT = 40       # Max number of preview images to save

# -- Detection pseudo-label settings ------------------------------------------
DET_PSEUDO_SCORE_THRESH = 0.18   # Min detector score for pseudo-label inclusion
DET_CLIP_VERIFY_THRESH = 0.16    # Min CLIP similarity for crop verification
DET_MIN_BOX_EDGE = 12            # Min box edge length in pixels
DET_EPOCHS = 6                   # Detection fine-tuning epochs
DET_PSEUDO_OVERSAMPLE = 4        # Oversample factor for Ghana pseudo-labels

# -- Grounding DINO settings (strong zero-shot teacher) -----------------------
GDINO_BOX_THRESH = 0.25    # Grounding DINO box confidence threshold
GDINO_TEXT_THRESH = 0.20   # Grounding DINO text matching threshold
GDINO_NMS_THRESH = 0.5     # NMS IoU threshold for deduplication
MAX_SOURCE_DET_IMAGES = 2500  # Max BDD100K source images for mixed training


In [5]:

# -- Utility functions ---------------------------------------------------------

def set_seed(seed):
    """Set random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

In [6]:
# -- Paths (Google Drive / Colab) ----------------------------------------------
# INPUT: Shared "CV" folder (read-only -- do NOT write here)
CV_FOLDER = "/content/drive/My Drive/Colab Notebooks/Computer Vision"
SHARED_CV = os.path.join(CV_FOLDER, "CV")

# Extracted frames zip -- from shared folder
FRAMES_ZIP = os.path.join(CV_FOLDER, "Extracted_frames.zip")
FRAMES_DIR = os.path.join(CV_FOLDER, "Extracted_frames")

# Model checkpoints from Sprint 1 (segmentation)
CITYSCAPE_MODEL_PATH = os.path.join(SHARED_CV, "deeplab_cityscape_best_model.pth")
AUGMENTED_MODEL_PATH = os.path.join(SHARED_CV, "augmented_city_scape_model.pth")

In [7]:
# Detection model from Sprint 2
DETECTOR_MODEL_PATH = os.path.join(SHARED_CV, "best_detector.pth")
MODEL_DETECT_SRC = os.path.join(SHARED_CV, "model_detect.py")

# OUTPUT: Your own Drive folder (write here -- safe, no overwrites)
SPRINT3_OUT = os.path.join(CV_FOLDER, "sprint3_output")
CATALOGUE_PATH = os.path.join(SPRINT3_OUT, "failure_catalogue.json")
PSEUDO_LABELS_DIR = os.path.join(SPRINT3_OUT, "pseudo_labels")
RETRAINED_MODEL_PATH = os.path.join(SPRINT3_OUT, "hard_negative_retrained_model.pth")
RETRAINED_DETECTOR_PATH = os.path.join(SPRINT3_OUT, "retrained_detector.pth")
REPORT_DIR = os.path.join(SPRINT3_OUT, "failure_report")
VIS_DIR = os.path.join(SPRINT3_OUT, "visualizations")

for directory in [SPRINT3_OUT, PSEUDO_LABELS_DIR, REPORT_DIR, VIS_DIR]:
    os.makedirs(directory, exist_ok=True)

print(f"Reading inputs from:  {SHARED_CV}")
print(f"Writing outputs to:   {SPRINT3_OUT}")

Reading inputs from:  /content/drive/My Drive/Colab Notebooks/Computer Vision/CV
Writing outputs to:   /content/drive/My Drive/Colab Notebooks/Computer Vision/sprint3_output


In [8]:
def robust_normalize(values, low_q=5, high_q=95):
    """Percentile-based normalization to [0, 1], robust to outliers."""
    values = np.asarray(values, dtype=np.float32)
    lo = np.percentile(values, low_q)
    hi = np.percentile(values, high_q)
    if hi - lo < 1e-6:
        return np.zeros_like(values, dtype=np.float32)
    return np.clip((values - lo) / (hi - lo), 0.0, 1.0)


def normalize_map(values):
    """Normalize a 2D array to [0, 1] by dividing by its maximum."""
    values = values.astype(np.float32)
    vmax = float(values.max())
    if vmax <= 1e-6:
        return np.zeros_like(values, dtype=np.float32)
    return values / vmax


def extract_clip_id(path):
    """Extract video clip identifier from a frame filename (e.g., 'clip01' from 'clip01_frame_042.jpg')."""
    stem = os.path.splitext(os.path.basename(path))[0]
    if "_frame_" in stem:
        return stem.split("_frame_")[0]
    return stem


def serialize_value(value):
    """Convert numpy/torch types to JSON-serializable Python types."""
    if isinstance(value, (np.floating, np.integer)):
        return value.item()
    if isinstance(value, torch.Tensor):
        return value.tolist()
    if isinstance(value, np.ndarray):
        return value.tolist()
    return value


def load_kaggle_credentials():
    """Load Kaggle API key from Colab secrets or environment."""
    try:
        from dotenv import load_dotenv
    except Exception:
        return

    load_dotenv()
    api_key = None
    if userdata is not None:
        try:
            api_key = userdata.get("KAGGLE_API_TOKEN")
        except Exception:
            api_key = None
    if api_key:
        os.environ["KAGGLE_KEY"] = api_key
        if not os.environ.get("KAGGLE_USERNAME"):
            os.environ["KAGGLE_USERNAME"] = "__token__"


def resolve_split_dirs(dataset_path, split):
    """Locate image/label directories for a dataset split (train/val)."""
    img_dir = os.path.join(dataset_path, split, "img")
    seg_dir = os.path.join(dataset_path, split, "label")
    if os.path.isdir(img_dir) and os.path.isdir(seg_dir):
        return img_dir, seg_dir

    for root, dirs, _ in os.walk(dataset_path):
        if split in dirs:
            candidate = os.path.join(root, split)
            contents = os.listdir(candidate)
            if "img" in contents and "label" in contents:
                return os.path.join(candidate, "img"), os.path.join(candidate, "label")

    raise FileNotFoundError(f"Could not resolve {split} directories under {dataset_path}")


def extract_road_mask_from_rgb(seg_np):
    """Convert an RGB segmentation label to a binary road mask using colour matching."""
    seg16 = seg_np.astype(np.int16)
    diff = np.abs(seg16 - np.array(ROAD_COLOR, dtype=np.int16))
    return (diff.max(axis=2) <= COLOR_TOLERANCE).astype(np.uint8)

In [9]:
# -- 1b. Unzip frames ---------------------------------------------------------

def ensure_frames():
    """Find or extract Ghanaian dashcam frames. Returns sorted list of JPG paths."""
    frame_paths = sorted(glob.glob(os.path.join(FRAMES_DIR, "*.jpg")))
    if not frame_paths:
        frame_paths = sorted(glob.glob(os.path.join(FRAMES_DIR, "**", "*.jpg"), recursive=True))

    if len(frame_paths) >= 100:
        print(f"Frames already extracted: {FRAMES_DIR} ({len(frame_paths)} JPGs found)")
        return frame_paths

    if os.path.isfile(FRAMES_ZIP):
        print("Unzipping extracted frames to Drive (one-time, may take a few minutes) ...")
        os.makedirs(FRAMES_DIR, exist_ok=True)
        with zipfile.ZipFile(FRAMES_ZIP, "r") as handle:
            handle.extractall(os.path.dirname(FRAMES_DIR))
        frame_paths = sorted(glob.glob(os.path.join(FRAMES_DIR, "**", "*.jpg"), recursive=True))
        print(f"Frames directory: {FRAMES_DIR} ({len(frame_paths)} JPGs extracted)")

    if not frame_paths:
        raise RuntimeError(
            f"No frames found in {FRAMES_DIR} and zip missing at {FRAMES_ZIP}. "
            "Upload Extraced_frames.zip or extract frames first."
        )
    return frame_paths


frame_paths = ensure_frames()
print(f"Total Ghanaian frames: {len(frame_paths)}")

Frames already extracted: /content/drive/My Drive/Colab Notebooks/Computer Vision/Extracted_frames (3797 JPGs found)
Total Ghanaian frames: 3797


In [10]:
# -- 1c. Load DeepLabV3+ architecture -----------------------------------------

REPO_DIR = "DeepLabV3Plus-Pytorch"
CHECKPOINT = "best_deeplabv3plus_resnet101_cityscapes_os16.pth"
GDRIVE_ID = "1t7TC8mxQaFECt4jutdq_NMnWxdm6B-Nb"

if not os.path.isdir(REPO_DIR):
    print("Cloning DeepLabV3Plus-Pytorch ...")
    subprocess.run(["git", "clone", "https://github.com/VainF/DeepLabV3Plus-Pytorch.git"], check=True)

if not os.path.isfile(CHECKPOINT):
    print("Downloading Cityscapes-pretrained checkpoint ...")
    gdown.download(id=GDRIVE_ID, output=CHECKPOINT, quiet=False)

sys.path.insert(0, REPO_DIR)
from network.modeling import deeplabv3plus_resnet101

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


def load_segmentation_model(checkpoint_path):
    """Load a 2-class road segmentation model from Sprint 1/2."""
    model = deeplabv3plus_resnet101(num_classes=19, output_stride=16)
    model.classifier.classifier[3] = nn.Conv2d(256, 2, kernel_size=1)
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state"])
    model = model.to(device).eval()
    return model


model_cityscape = load_segmentation_model(CITYSCAPE_MODEL_PATH)
model_augmented = load_segmentation_model(AUGMENTED_MODEL_PATH)
print("[OK] Both segmentation models loaded")


Cloning DeepLabV3Plus-Pytorch ...


Downloading...
From (original): https://drive.google.com/uc?id=1t7TC8mxQaFECt4jutdq_NMnWxdm6B-Nb
From (redirected): https://drive.google.com/uc?id=1t7TC8mxQaFECt4jutdq_NMnWxdm6B-Nb&confirm=t&uuid=b531aa23-550e-4811-baba-bb1cc63bcd17
To: /content/best_deeplabv3plus_resnet101_cityscapes_os16.pth
100%|██████████| 471M/471M [00:07<00:00, 61.5MB/s]


Using device: cuda
Downloading: "https://download.pytorch.org/models/resnet101-5d3b4d8f.pth" to /root/.cache/torch/hub/checkpoints/resnet101-5d3b4d8f.pth


100%|██████████| 170M/170M [00:00<00:00, 392MB/s]


[OK] Both segmentation models loaded


In [11]:

# -- 1d. Load detection model (Sprint 2) --------------------------------------

if os.path.isfile(MODEL_DETECT_SRC) and not os.path.isfile("model_detect.py"):
    shutil.copy(MODEL_DETECT_SRC, "model_detect.py")
    print("Copied model_detect.py from shared folder")

from model_detect import (
    BDD100K_CLASS_MAP,
    DETECT_CLASSES,
    NUM_DETECT_CLASSES,
    RoadAwareDetector,
    road_aware_filter,
)


def load_detection_model(ckpt_path):
    """Load the Sprint 2 anchor-based FPN detector."""
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    img_h = ckpt.get("img_h", 448)
    img_w = ckpt.get("img_w", ckpt.get("img_size", 800))
    model = RoadAwareDetector(num_classes=NUM_DETECT_CLASSES, pretrained_backbone=False)
    model.load_state_dict(ckpt["model_state"])
    model.to(device).eval()
    print(
        f"  Detector loaded: epoch={ckpt.get('epoch', '?')}, "
        f"mAP@0.5={ckpt.get('mAP_50', '?')}, input={img_h}x{img_w}"
    )
    return model, img_h, img_w


model_detector, DET_IMG_H, DET_IMG_W = load_detection_model(DETECTOR_MODEL_PATH)
print("[OK] Detection model loaded")

Copied model_detect.py from shared folder
  Detector loaded: epoch=29, mAP@0.5=0.183936618399015, input=448x800
[OK] Detection model loaded


In [12]:
# -- Image transforms (shared across all models) ------------------------------

to_tensor = transforms.ToTensor()
seg_normalize = transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
det_normalize = transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
seg_color_jitter = transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.20)
det_color_jitter = transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10)


def image_to_seg_tensor(img):
    return seg_normalize(to_tensor(img))


def image_to_det_tensor(img):
    """Convert a PIL image to a normalized tensor for detection."""
    return det_normalize(to_tensor(img))


def resize_for_segmentation(img):
    """Resize a PIL image to the segmentation input size."""
    return img.resize((IMG_SIZE[1], IMG_SIZE[0]), Image.BILINEAR)


def resize_for_detection(img):
    """Resize a PIL image to the detection input size."""
    return img.resize((DET_IMG_W, DET_IMG_H), Image.BILINEAR)


class FrameDataset(Dataset):
    """Simple dataset for loading extracted Ghanaian dashcam frames."""
    def __init__(self, paths):
        self.paths = paths

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        img = resize_for_segmentation(img)
        return image_to_seg_tensor(img), idx


frame_dataset = FrameDataset(frame_paths)
frame_loader = DataLoader(frame_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


def run_inference(model, loader, desc="Inference", use_tta=False):
    """Run segmentation inference. If use_tta=True, average original + horizontally flipped."""
    all_probs = []
    all_preds = []
    all_entropy = []
    frame_metrics = []

    with torch.no_grad():
        for imgs, indices in tqdm(loader, desc=desc):
            imgs = imgs.to(device)

            # Forward pass on original
            output = model(imgs)
            probs = torch.softmax(output, dim=1)

            if use_tta:
                # Forward pass on horizontally flipped images
                imgs_flip = torch.flip(imgs, [-1])
                output_flip = model(imgs_flip)
                probs_flip = torch.softmax(output_flip, dim=1)
                probs_flip = torch.flip(probs_flip, [-1])  # flip back
                probs = (probs + probs_flip) / 2.0

            road_prob = probs[:, 1].cpu().numpy()
            road_pred = (road_prob >= 0.5).astype(np.uint8)
            entropy = -(probs * torch.log(probs + 1e-8)).sum(dim=1).cpu().numpy()

            for batch_idx in range(imgs.shape[0]):
                frame_idx = indices[batch_idx].item()
                pred = road_pred[batch_idx]
                prob = road_prob[batch_idx]
                ent = entropy[batch_idx]
                uncertain_mask = (prob > 0.3) & (prob < 0.7)

                frame_metrics.append(
                    {
                        "idx": frame_idx,
                        "path": frame_paths[frame_idx],
                        "mean_entropy": float(ent.mean()),
                        "road_ratio": float(pred.mean()),
                        "mean_road_conf": float(prob[pred.astype(bool)].mean()) if pred.any() else 0.0,
                        "uncertain_ratio": float(uncertain_mask.mean()),
                    }
                )

            all_probs.append(road_prob)
            all_preds.append(road_pred)
            all_entropy.append(entropy)

    return (
        np.concatenate(all_probs, axis=0),
        np.concatenate(all_preds, axis=0),
        np.concatenate(all_entropy, axis=0),
        frame_metrics,
    )


print("\n" + "=" * 60)
print("SECTION 2: Segmentation Inference on Ghanaian Frames")
print("=" * 60)

print("\nRunning Cityscapes-trained model (with TTA) ...")
probs_city, preds_city, entropy_city, metrics_city = run_inference(
    model_cityscape, frame_loader, desc="Cityscapes model + TTA", use_tta=True
)

print("\nRunning Ghana-augmented model (with TTA) ...")
probs_aug, preds_aug, entropy_aug, metrics_aug = run_inference(
    model_augmented, frame_loader, desc="Augmented model + TTA", use_tta=True
)

disagreement = (preds_city != preds_aug).astype(np.float32)
per_frame_disagreement = disagreement.reshape(len(frame_paths), -1).mean(axis=1)


SECTION 2: Segmentation Inference on Ghanaian Frames

Running Cityscapes-trained model (with TTA) ...


Cityscapes model + TTA: 100%|██████████| 475/475 [03:28<00:00,  2.28it/s]



Running Ghana-augmented model (with TTA) ...


Augmented model + TTA: 100%|██████████| 475/475 [00:56<00:00,  8.39it/s]


In [13]:
# -- 2b. Run detection model on Ghanaian frames --------------------------------

def run_detection_inference(det_model, road_masks):
    """Run the Sprint 2 detector on all frames with road-aware filtering."""
    print("\nRunning detection model on Ghanaian frames ...")
    det_results = []
    with torch.no_grad():
        for idx, img_path in enumerate(tqdm(frame_paths, desc="Detection inference")):
            img_pil = Image.open(img_path).convert("RGB")
            det_img = resize_for_detection(img_pil)
            det_tensor = image_to_det_tensor(det_img).unsqueeze(0).to(device)
            results = det_model(det_tensor)[0]

            boxes = results["boxes"]
            scores = results["scores"]
            labels = results["labels"]

            if boxes.shape[0] > 0:
                boxes, scores, labels = road_aware_filter(
                    boxes,
                    scores,
                    labels,
                    torch.from_numpy(road_masks[idx].astype(np.uint8)),
                    DET_IMG_H,
                    DET_IMG_W,
                )

            keep = scores > 0.3
            boxes, scores, labels = boxes[keep], scores[keep], labels[keep]

            class_counts = {
                DETECT_CLASSES[class_idx]: int((labels == class_idx).sum())
                for class_idx in range(NUM_DETECT_CLASSES)
            }

            det_results.append(
                {
                    "n_detections": int(len(boxes)),
                    "mean_det_score": float(scores.mean()) if len(scores) else 0.0,
                    "class_counts": class_counts,
                    "boxes": boxes.cpu(),
                    "scores": scores.cpu(),
                    "labels": labels.cpu(),
                }
            )
    return det_results


det_results_per_frame = run_detection_inference(model_detector, preds_aug)


Running detection model on Ghanaian frames ...


Detection inference: 100%|██████████| 3797/3797 [02:47<00:00, 22.71it/s]


In [14]:

# -- 2c. Merge segmentation + detection metrics into frame metadata -----------

total_dets = sum(item["n_detections"] for item in det_results_per_frame)
print(f"Detection summary over {len(frame_paths)} Ghanaian frames:")
print(f"  Total detections: {total_dets}")
print(f"  Mean detections/frame: {total_dets / len(frame_paths):.1f}")
print(f"  Frames with 0 detections: {sum(1 for item in det_results_per_frame if item['n_detections'] == 0)}")

frame_metadata = []
for idx in range(len(frame_paths)):
    meta = metrics_aug[idx].copy()
    meta["entropy_city"] = metrics_city[idx]["mean_entropy"]
    meta["disagreement"] = float(per_frame_disagreement[idx])
    meta["n_detections"] = det_results_per_frame[idx]["n_detections"]
    meta["mean_det_score"] = det_results_per_frame[idx]["mean_det_score"]
    meta["class_counts"] = det_results_per_frame[idx]["class_counts"]
    meta["uncertainty_score"] = (
        0.4 * meta["mean_entropy"] + 0.3 * meta["uncertain_ratio"] + 0.3 * meta["disagreement"]
    )
    frame_metadata.append(meta)

uncertainty_values = np.array([meta["uncertainty_score"] for meta in frame_metadata], dtype=np.float32)
uncertainty_norm = robust_normalize(uncertainty_values)
for idx, meta in enumerate(frame_metadata):
    meta["uncertainty_norm"] = float(uncertainty_norm[idx])

print(f"\nProcessed {len(frame_metadata)} frames")
for meta in sorted(frame_metadata, key=lambda item: item["uncertainty_score"], reverse=True)[:5]:
    print(
        f"  {os.path.basename(meta['path'])}: uncertainty={meta['uncertainty_score']:.4f}, "
        f"disagreement={meta['disagreement']:.4f}, detections={meta['n_detections']}"
    )


Detection summary over 3797 Ghanaian frames:
  Total detections: 2102
  Mean detections/frame: 0.6
  Frames with 0 detections: 2465

Processed 3797 frames
  20260221_103800_1_frame_433.jpg: uncertainty=0.1472, disagreement=0.1832, detections=0
  20260221_103800_1_frame_401.jpg: uncertainty=0.1455, disagreement=0.1540, detections=0
  20260221_103800_1_frame_428.jpg: uncertainty=0.1439, disagreement=0.1247, detections=0
  20260221_103800_1_frame_429.jpg: uncertainty=0.1423, disagreement=0.1352, detections=0
  20260221_103800_1_frame_432.jpg: uncertainty=0.1416, disagreement=0.1814, detections=0


In [15]:
"""
===============================================================================
SECTION 3: CLIP-Based Failure Mining with Text Queries
===============================================================================
Use CLIP to score frames against failure-pattern text prompts + object-presence
prompts. Select a balanced, diverse set of hard negatives across categories.
"""

print("\n" + "=" * 60)
print("SECTION 3: CLIP-Based Failure Mining")
print("=" * 60)

import open_clip

# -- 3a. Define failure-pattern prompts ----------------------------------------
# Each category has multiple text prompts describing road conditions that break
# segmentation (shadows, unpaved, occlusion, etc.)


FAILURE_PROMPTS = {
    "shadow_confusion": [
        "road covered in dark shadows that look like road edges",
        "shadows on asphalt that resemble potholes or cracks",
    ],
    "dust_texture": [
        "dusty unpaved road surface with no clear edges",
        "sandy or gravel road with confusing texture",
    ],
    "road_degradation": [
        "road with potholes and dark asphalt patches",
        "damaged road with missing lane markings",
    ],
    "pedestrian_occlusion": [
        "pedestrians walking on or very close to the road",
        "people partially occluded by vehicles on a busy road",
    ],
    "market_clutter": [
        "crowded market area with people and vehicles on the road",
        "informal roadside stalls and vendors near the road",
    ],
    "low_light": [
        "road at dusk or in low light conditions",
        "overexposed or glaring road surface",
    ],
    "speed_bumps": [
        "road with speed bumps or rumble strips",
        "raised obstacles on the road surface",
    ],
    "informal_obstacles": [
        "rocks or branches placed on the road as markers",
        "construction debris or barriers on the road",
    ],
    "wet_surface": [
        "wet or muddy road surface with reflections",
        "road puddles that distort surface appearance",
    ],
    "vehicle_confusion": [
        "vehicles parked on the road edge blending with road",
        "motorcycle or okada on an unpaved road",
    ],
}

# -- Object-presence prompts for detection class tagging ----------------------
# Used to tag frames with which object classes are likely visible

OBJECT_PROMPTS = {
    "car": ["car driving on a road", "taxi or vehicle on a busy street"],
    "bus": ["bus on a road in a city", "public transport bus on a street"],
    "truck": ["truck or lorry on a highway", "heavy vehicle on a road"],
    "pedestrian": ["pedestrians walking along the road", "people crossing a street"],
    "rider": ["person riding a motorcycle", "motorcycle rider on a road"],
    "motorcycle": ["motorcycle parked or moving on a road", "okada on a dusty road"],
    "bicycle": ["person riding a bicycle on a road", "bicycle on a city street"],
    "traffic sign": ["traffic signs along a road", "road signs visible from a car"],
    "traffic light": ["traffic lights at an intersection", "signal lights on a road"],
}



SECTION 3: CLIP-Based Failure Mining


In [16]:
# -- 3b. CLIP dataset and model loading ----------------------------------------

class CLIPFrameDataset(Dataset):
    """Dataset for batched CLIP image encoding."""
    def __init__(self, paths, preprocess):
        self.paths = paths
        self.preprocess = preprocess

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        return self.preprocess(img), idx


# Load CLIP ViT-B/32 for text-image similarity scoring
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k"
)
clip_model = clip_model.to(device).eval()
tokenizer = open_clip.get_tokenizer("ViT-B-32")

# Flatten all prompts (failure categories + object presence) for batch encoding
all_prompts = []
prompt_categories = []
for category, prompts in FAILURE_PROMPTS.items():
    for prompt in prompts:
        all_prompts.append(prompt)
        prompt_categories.append(category)

obj_prompt_offset = len(all_prompts)
for category, prompts in OBJECT_PROMPTS.items():
    for prompt in prompts:
        all_prompts.append(prompt)
        prompt_categories.append(f"obj_{category}")

print(f"Defined {len(all_prompts)} prompts")

with torch.no_grad():
    text_tokens = tokenizer(all_prompts).to(device)
    text_features = clip_model.encode_text(text_tokens)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Defined 38 prompts


In [17]:
# -- 3c. Compute per-frame CLIP similarities ----------------------------------

clip_dataset = CLIPFrameDataset(frame_paths, clip_preprocess)
clip_loader = DataLoader(clip_dataset, batch_size=32, shuffle=False, num_workers=2)

clip_scores = np.zeros((len(frame_paths), len(all_prompts)), dtype=np.float32)
clip_image_features = np.zeros((len(frame_paths), int(text_features.shape[-1])), dtype=np.float32)

with torch.no_grad():
    for imgs, indices in tqdm(clip_loader, desc="CLIP encoding"):
        imgs = imgs.to(device)
        image_features = clip_model.encode_image(imgs)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        similarity = (image_features @ text_features.T).cpu().numpy()
        image_features_np = image_features.cpu().numpy()
        for batch_pos, frame_idx in enumerate(indices):
            idx = frame_idx.item()
            clip_scores[idx] = similarity[batch_pos]
            clip_image_features[idx] = image_features_np[batch_pos]

# Aggregate per-category max scores and per-object-class max scores
category_names = list(FAILURE_PROMPTS.keys())
category_scores = np.zeros((len(frame_paths), len(category_names)), dtype=np.float32)
category_score_norms = {}
for category_idx, category in enumerate(category_names):
    prompt_indices = [i for i, value in enumerate(prompt_categories) if value == category]
    category_scores[:, category_idx] = clip_scores[:, prompt_indices].max(axis=1)
    category_score_norms[category] = robust_normalize(category_scores[:, category_idx])

max_clip_score = category_scores.max(axis=1)
max_clip_norm = robust_normalize(max_clip_score)

object_class_names = list(OBJECT_PROMPTS.keys())
object_scores = np.zeros((len(frame_paths), len(object_class_names)), dtype=np.float32)
for class_idx, class_name in enumerate(object_class_names):
    prompt_indices = [
        i
        for i in range(obj_prompt_offset, len(all_prompts))
        if prompt_categories[i] == f"obj_{class_name}"
    ]
    object_scores[:, class_idx] = clip_scores[:, prompt_indices].max(axis=1)

for meta in frame_metadata:
    idx = meta["idx"]
    meta["clip_scores"] = {
        category: float(category_scores[idx, category_idx])
        for category_idx, category in enumerate(category_names)
    }
    meta["max_clip_score"] = float(max_clip_score[idx])
    meta["top_failure_category"] = category_names[int(category_scores[idx].argmax())]
    meta["failure_score"] = float(0.55 * uncertainty_norm[idx] + 0.45 * max_clip_norm[idx])
    meta["category_selection_scores"] = {
        category: float(0.60 * meta["failure_score"] + 0.40 * category_score_norms[category][idx])
        for category in category_names
    }
    meta["clip_object_scores"] = {
        class_name: float(object_scores[idx, class_idx])
        for class_idx, class_name in enumerate(object_class_names)
    }
    median_obj = float(np.median(object_scores[idx]))
    meta["clip_likely_classes"] = [
        class_name
        for class_idx, class_name in enumerate(object_class_names)
        if object_scores[idx, class_idx] > median_obj + 0.01
    ]

print("\nTop-10 failure candidates before balancing:")
for meta in sorted(frame_metadata, key=lambda item: item["failure_score"], reverse=True)[:10]:
    print(
        f"  {os.path.basename(meta['path'])}: failure={meta['failure_score']:.4f}, "
        f"category={meta['top_failure_category']}, clip={meta['max_clip_score']:.4f}, "
        f"uncertainty={meta['uncertainty_score']:.4f}"
    )

CLIP encoding: 100%|██████████| 119/119 [00:33<00:00,  3.59it/s]


Top-10 failure candidates before balancing:
  20260221_103800_1_frame_759.jpg: failure=1.0000, category=vehicle_confusion, clip=0.3177, uncertainty=0.1145
  20260221_103800_1_frame_761.jpg: failure=1.0000, category=vehicle_confusion, clip=0.3231, uncertainty=0.1100
  20260221_103800_1_frame_762.jpg: failure=1.0000, category=vehicle_confusion, clip=0.3309, uncertainty=0.1212
  20260221_103800_1_frame_763.jpg: failure=1.0000, category=vehicle_confusion, clip=0.3249, uncertainty=0.1107
  20260221_103800_2_frame_263.jpg: failure=1.0000, category=vehicle_confusion, clip=0.3235, uncertainty=0.1134
  20260221_103800_2_frame_269.jpg: failure=1.0000, category=vehicle_confusion, clip=0.3160, uncertainty=0.1106
  20260221_103800_2_frame_268.jpg: failure=0.9976, category=vehicle_confusion, clip=0.3165, uncertainty=0.1096
  20260221_103800_1_frame_760.jpg: failure=0.9897, category=vehicle_confusion, clip=0.3091, uncertainty=0.1276
  20260221_103800_2_frame_267.jpg: failure=0.9848, category=vehicle

In [18]:
# -- 3d. Balanced, diverse hard-negative selection ----------------------------

def greedy_diverse_pick(candidates, score_key, total_needed, selected_ids, clip_counts):
    """Greedily pick candidates that maximize score + embedding diversity."""
    chosen = []
    pool = [candidate for candidate in candidates if candidate["idx"] not in selected_ids]
    if not pool or total_needed <= 0:
        return chosen

    pool.sort(key=lambda item: item[score_key], reverse=True)
    while pool and len(chosen) < total_needed:
        best_item = None
        best_value = -1e9
        for candidate in pool:
            clip_id = extract_clip_id(candidate["path"])
            if clip_counts[clip_id] >= MAX_FRAMES_PER_CLIP:
                continue
            if not chosen:
                diversity_bonus = 1.0
            else:
                candidate_feat = clip_image_features[candidate["idx"]]
                diversity_bonus = min(
                    1.0 - float(np.dot(candidate_feat, clip_image_features[pick["idx"]]))
                    for pick in chosen
                )
            value = float(candidate[score_key]) + 0.30 * diversity_bonus
            if value > best_value:
                best_value = value
                best_item = candidate
        if best_item is None:
            break
        chosen.append(best_item)
        selected_ids.add(best_item["idx"])
        clip_counts[extract_clip_id(best_item["path"])] += 1
        pool = [candidate for candidate in pool if candidate["idx"] != best_item["idx"]]
    return chosen


def select_balanced_hard_negatives(metadata, total_count):
    """Select hard negatives with balanced category representation and diversity.

    Allocates an equal quota per failure category, then fills remaining slots
    from the overall top-scoring frames. Uses greedy_diverse_pick to avoid
    near-duplicate frames from the same video clip.
    """
    selected_ids = set()
    clip_counts = defaultdict(int)
    selected = []
    base_quota = max(1, total_count // len(category_names))

    for category in category_names:
        category_candidates = [
            candidate
            for candidate in metadata
            if candidate["top_failure_category"] == category
        ]
        if not category_candidates:
            continue
        for candidate in category_candidates:
            candidate["selection_score"] = candidate["category_selection_scores"][category]
        category_candidates.sort(key=lambda item: item["selection_score"], reverse=True)
        quota = min(base_quota, len(category_candidates))
        selected.extend(
            greedy_diverse_pick(
                category_candidates[: max(quota * 6, quota)],
                "selection_score",
                quota,
                selected_ids,
                clip_counts,
            )
        )

    remaining = total_count - len(selected)
    if remaining > 0:
        fallback_pool = sorted(metadata, key=lambda item: item["failure_score"], reverse=True)
        for candidate in fallback_pool:
            candidate["selection_score"] = candidate["failure_score"]
        selected.extend(
            greedy_diverse_pick(
                fallback_pool[: max(remaining * 8, remaining)],
                "selection_score",
                remaining,
                selected_ids,
                clip_counts,
            )
        )

    selected.sort(key=lambda item: item["failure_score"], reverse=True)
    return selected[:total_count]


hard_negatives = select_balanced_hard_negatives(frame_metadata, HARD_NEG_COUNT)
hn_cats = defaultdict(int)
for item in hard_negatives:
    hn_cats[item["top_failure_category"]] += 1

print(f"\nBalanced hard-negative selection: {len(hard_negatives)} frames")
for category, count in sorted(hn_cats.items(), key=lambda pair: (-pair[1], pair[0])):
    print(f"  {category}: {count} frames ({100.0 * count / len(hard_negatives):.1f}%)")

print("\nObject-presence tagging:")
for class_name in object_class_names:
    count = sum(1 for item in hard_negatives if class_name in item.get("clip_likely_classes", []))
    print(f"  {class_name}: {count}")

del clip_model, text_features, text_tokens
torch.cuda.empty_cache()


Balanced hard-negative selection: 72 frames
  dust_texture: 30 frames (41.7%)
  road_degradation: 30 frames (41.7%)
  vehicle_confusion: 5 frames (6.9%)
  pedestrian_occlusion: 4 frames (5.6%)
  low_light: 1 frames (1.4%)
  market_clutter: 1 frames (1.4%)
  speed_bumps: 1 frames (1.4%)

Object-presence tagging:
  car: 44
  bus: 1
  truck: 61
  pedestrian: 2
  rider: 2
  motorcycle: 72
  bicycle: 5
  traffic sign: 49
  traffic light: 3


In [19]:
"""
===============================================================================
SECTION 4: BLIP Captioning of Failure Cases
===============================================================================
Caption top hard-negative frames with BLIP to produce human-readable
descriptions for the failure analysis report.
"""

print("\n" + "=" * 60)
print("SECTION 4: BLIP Captioning of Failure Cases")
print("=" * 60)

from transformers import BlipForConditionalGeneration, BlipProcessor

blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
).to(device).eval()

caption_limit = min(len(hard_negatives), 180)
caption_targets = hard_negatives[:caption_limit]
print(f"Generating captions for {caption_limit} selected hard negatives ...")

for meta in tqdm(caption_targets, desc="BLIP captioning"):
    img = Image.open(meta["path"]).convert("RGB")
    inputs = blip_processor(img, return_tensors="pt").to(device)
    with torch.no_grad():
        out = blip_model.generate(**inputs, max_new_tokens=50)
    meta["blip_caption"] = blip_processor.decode(out[0], skip_special_tokens=True)

    cond_inputs = blip_processor(img, text="a photo of a road with", return_tensors="pt").to(device)
    with torch.no_grad():
        out_cond = blip_model.generate(**cond_inputs, max_new_tokens=50)
    meta["blip_road_caption"] = blip_processor.decode(out_cond[0], skip_special_tokens=True)

for meta in hard_negatives[:5]:
    print(f"  {os.path.basename(meta['path'])}:")
    print(f"    Caption: {meta.get('blip_caption', 'N/A')}")
    print(f"    Road:    {meta.get('blip_road_caption', 'N/A')}")
    print(f"    Category: {meta['top_failure_category']}")

del blip_model, blip_processor
torch.cuda.empty_cache()



SECTION 4: BLIP Captioning of Failure Cases


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

...

Generating captions for 72 selected hard negatives ...


BLIP captioning:   1%|▏         | 1/72 [00:00<00:19,  3.61it/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

BLIP captioning: 100%|██████████| 72/72 [00:12<00:00,  5.81it/s]


  20260221_103800_2_frame_971.jpg:
    Caption: a street with houses and buildings on both sides
    Road:    a photo of a road with houses and buildings
    Category: dust_texture
  20260221_104249_1_frame_098.jpg:
    Caption: a person riding a motorcycle down a street
    Road:    a photo of a road with a person riding a bike
    Category: vehicle_confusion
  20260221_103800_2_frame_972.jpg:
    Caption: a street with houses and buildings on both sides
    Road:    a photo of a road with houses and buildings
    Category: dust_texture
  20260221_104249_1_frame_097.jpg:
    Caption: a person riding a bike down a street
    Road:    a photo of a road with a person riding a bike
    Category: vehicle_confusion
  20260221_103800_1_frame_681.jpg:
    Caption: a road with a bunch of trees on the side
    Road:    a photo of a road with a building in the background
    Category: road_degradation


In [20]:
"""
===============================================================================
SECTION 5: Confidence-Masked Pseudo-Labelling
===============================================================================
Generate per-pixel pseudo-labels from teacher ensemble consensus. Regions where
teachers disagree or are uncertain are masked (IGNORE_INDEX=255) and given zero
weight. This avoids training on noisy labels.
"""

print("\n" + "=" * 60)
print("SECTION 5: Confidence-Masked Pseudo-Labelling")
print("=" * 60)


def postprocess_road_mask(mask):
    """Clean up a binary road mask: close gaps, remove small noise, keep
    only the largest connected component touching the bottom of the image."""
    mask = mask.astype(np.uint8)
    if mask.sum() == 0:
        return mask

    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((7, 7), np.uint8), iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8), iterations=1)

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if num_labels <= 1:
        return mask

    bottom_labels = [label for label in np.unique(labels[-1]) if label != 0]
    if bottom_labels:
        keep = np.zeros_like(mask)
        for label in bottom_labels:
            keep[labels == label] = 1
        if keep.sum() > 0:
            return keep.astype(np.uint8)

    largest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    keep = np.zeros_like(mask)
    keep[labels == largest] = 1
    return keep.astype(np.uint8)


def save_pseudo_preview(hard_mask, path):
    """Save a visual preview of a pseudo-label (road=purple, ignored=white)."""
    preview = np.zeros((hard_mask.shape[0], hard_mask.shape[1], 3), dtype=np.uint8)
    preview[hard_mask == 1] = ROAD_COLOR
    preview[hard_mask == IGNORE_INDEX] = (255, 255, 255)
    Image.fromarray(preview).save(path)


def generate_pseudo_labels(selected_frames, teacher_probs, teacher_entropy, round_idx):
    """Generate confidence-masked pseudo-labels from teacher ensemble.

    For each frame, averages probability maps from all teachers, then masks
    out pixels where teachers disagree or entropy is too high. Also computes
    per-pixel weights and a focus map for hard-crop selection.
    """
    round_preview_paths = []
    for preview_idx, meta in enumerate(
        tqdm(selected_frames, desc=f"Pseudo-labelling round {round_idx + 1}")
    ):
        idx = meta["idx"]
        prob_stack = np.stack([item[idx] for item in teacher_probs], axis=0)
        entropy_stack = np.stack([item[idx] for item in teacher_entropy], axis=0)

        mean_prob = prob_stack.mean(axis=0).astype(np.float32)
        mean_entropy = entropy_stack.mean(axis=0).astype(np.float32)

        pred_stack = prob_stack >= 0.5
        vote_fraction = pred_stack.mean(axis=0).astype(np.float32)
        agreement_ratio = np.maximum(vote_fraction, 1.0 - vote_fraction)
        margin = np.abs(mean_prob - 0.5) * 2.0

        if prob_stack.shape[0] == 2:
            valid = (agreement_ratio == 1.0) | (margin >= PSEUDO_DISAGREE_MARGIN)
        else:
            valid = (agreement_ratio == 1.0) | ((agreement_ratio >= 2.0 / 3.0) & (margin >= 0.18))

        valid &= normalize_map(mean_entropy) < 0.90

        hard_mask = np.full(mean_prob.shape, IGNORE_INDEX, dtype=np.uint8)
        hard_mask[valid] = (mean_prob[valid] >= 0.5).astype(np.uint8)

        cleaned_road = postprocess_road_mask((hard_mask == 1).astype(np.uint8))
        removed_road = (hard_mask == 1) & (cleaned_road == 0)
        hard_mask[removed_road] = IGNORE_INDEX

        pixel_weight = 0.6 * margin + 0.4 * np.clip((agreement_ratio - 0.5) * 2.0, 0.0, 1.0)
        pixel_weight = np.clip(pixel_weight, PSEUDO_WEIGHT_FLOOR, 1.0).astype(np.float32)
        pixel_weight[hard_mask == IGNORE_INDEX] = 0.0

        boundary = cv2.morphologyEx(cleaned_road, cv2.MORPH_GRADIENT, np.ones((5, 5), np.uint8))
        focus_map = (
            0.40 * normalize_map(mean_entropy)
            + 0.30 * (1.0 - agreement_ratio)
            + 0.20 * boundary.astype(np.float32)
            + 0.10 * (hard_mask == IGNORE_INDEX).astype(np.float32)
        ).astype(np.float32)

        name = os.path.splitext(os.path.basename(meta["path"]))[0]
        preview_path = os.path.join(PSEUDO_LABELS_DIR, f"round{round_idx + 1}_{name}_label.png")
        save_pseudo_preview(hard_mask, preview_path)
        if preview_idx < PSEUDO_PREVIEW_LIMIT:
            round_preview_paths.append(preview_path)

        meta["pseudo_label_path"] = preview_path
        meta["pseudo_hard"] = hard_mask
        meta["pseudo_soft"] = mean_prob
        meta["pseudo_weight"] = pixel_weight
        meta["focus_map"] = focus_map
        meta["consensus_agreement"] = float(agreement_ratio.mean())
        meta["pseudo_valid_ratio"] = float((hard_mask != IGNORE_INDEX).mean())
        meta["pseudo_mean_weight"] = float(pixel_weight[pixel_weight > 0].mean()) if (pixel_weight > 0).any() else 0.0
        meta["ignored_pixel_ratio"] = float((hard_mask == IGNORE_INDEX).mean())
    return {
        "preview_paths": round_preview_paths,
        "mean_valid_ratio": float(np.mean([meta["pseudo_valid_ratio"] for meta in selected_frames])),
        "mean_ignored_ratio": float(np.mean([meta["ignored_pixel_ratio"] for meta in selected_frames])),
        "mean_weight": float(np.mean([meta["pseudo_mean_weight"] for meta in selected_frames])),
    }


def build_hard_crop_specs(selected_frames, per_frame):
    """Select focus crops per frame from regions with highest uncertainty/disagreement."""
    specs = []
    crop_h, crop_w = HARD_CROP_SIZE
    img_h, img_w = IMG_SIZE
    for meta in selected_frames:
        focus = meta["focus_map"].copy()
        if focus.max() <= 1e-6:
            focus[:] = 1.0
        for _ in range(per_frame):
            y, x = np.unravel_index(int(focus.argmax()), focus.shape)
            x1 = int(np.clip(x - crop_w // 2, 0, img_w - crop_w))
            y1 = int(np.clip(y - crop_h // 2, 0, img_h - crop_h))
            x2 = x1 + crop_w
            y2 = y1 + crop_h
            specs.append({"frame_idx": meta["idx"], "box": (x1, y1, x2, y2)})
            focus[max(0, y1 - 20):min(img_h, y2 + 20), max(0, x1 - 20):min(img_w, x2 + 20)] = 0.0
    return specs


def convert_seg_sample(image_pil, hard_mask, soft_mask, weight_mask):
    """Convert a PIL image + masks to the dictionary format expected by the training loop."""
    return {
        "image": image_to_seg_tensor(image_pil),
        "hard_target": torch.from_numpy(hard_mask.astype(np.int64)),
        "soft_target": torch.from_numpy(soft_mask.astype(np.float32)),
        "weight": torch.from_numpy(weight_mask.astype(np.float32)),
    }



SECTION 5: Confidence-Masked Pseudo-Labelling


In [21]:

# -- 5b. Datasets for segmentation training -----------------------------------

class CityscapesRoad(Dataset):
    """Cityscapes road segmentation dataset (from Sprint 1)."""
    def __init__(self, img_dir, seg_dir, augment=True):
        self.images = sorted(glob.glob(os.path.join(img_dir, "*.*")))
        self.labels = sorted(glob.glob(os.path.join(seg_dir, "*.*")))
        assert len(self.images) == len(self.labels)
        self.augment = augment
        print(f"Cityscapes ({'train' if augment else 'eval'}): {len(self.images)} image/label pairs")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert("RGB")
        seg = Image.open(self.labels[idx]).convert("RGB")

        if self.augment and random.random() > 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
            seg = seg.transpose(Image.FLIP_LEFT_RIGHT)

        img = resize_for_segmentation(img)
        if self.augment and random.random() > 0.3:
            img = seg_color_jitter(img)

        seg = seg.resize((IMG_SIZE[1], IMG_SIZE[0]), Image.NEAREST)
        hard_mask = extract_road_mask_from_rgb(np.array(seg)).astype(np.uint8)
        soft_mask = hard_mask.astype(np.float32)
        weight_mask = np.ones_like(soft_mask, dtype=np.float32)
        return convert_seg_sample(img, hard_mask, soft_mask, weight_mask)


class HardNegativeDataset(Dataset):
    """Dataset of mined hard negative frames with pseudo-labels (full frames)."""
    def __init__(self, metadata, augment=True):
        self.metadata = metadata
        self.augment = augment
        print(f"Hard negatives (full frames): {len(self.metadata)}")

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        meta = self.metadata[idx]
        img = Image.open(meta["path"]).convert("RGB")
        img = resize_for_segmentation(img)

        hard_mask = meta["pseudo_hard"].copy()
        soft_mask = meta["pseudo_soft"].copy()
        weight_mask = meta["pseudo_weight"].copy()

        if self.augment and random.random() > 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
            hard_mask = np.fliplr(hard_mask).copy()
            soft_mask = np.fliplr(soft_mask).copy()
            weight_mask = np.fliplr(weight_mask).copy()

        if self.augment and random.random() > 0.3:
            img = seg_color_jitter(img)

        return convert_seg_sample(img, hard_mask, soft_mask, weight_mask)


class HardCropDataset(Dataset):
    """Dataset of focus crops around uncertain/boundary regions of hard negatives."""
    def __init__(self, metadata, crop_specs, augment=True):
        self.by_idx = {meta["idx"]: meta for meta in metadata}
        self.crop_specs = crop_specs
        self.augment = augment
        print(f"Hard negatives (crops): {len(self.crop_specs)}")

    def __len__(self):
        return len(self.crop_specs)

    def __getitem__(self, idx):
        spec = self.crop_specs[idx]
        meta = self.by_idx[spec["frame_idx"]]
        x1, y1, x2, y2 = spec["box"]

        img = Image.open(meta["path"]).convert("RGB")
        img = resize_for_segmentation(img)
        img = img.crop((x1, y1, x2, y2)).resize((IMG_SIZE[1], IMG_SIZE[0]), Image.BILINEAR)

        hard_mask = cv2.resize(
            meta["pseudo_hard"][y1:y2, x1:x2],
            (IMG_SIZE[1], IMG_SIZE[0]),
            interpolation=cv2.INTER_NEAREST,
        )
        soft_mask = cv2.resize(
            meta["pseudo_soft"][y1:y2, x1:x2],
            (IMG_SIZE[1], IMG_SIZE[0]),
            interpolation=cv2.INTER_LINEAR,
        )
        weight_mask = cv2.resize(
            meta["pseudo_weight"][y1:y2, x1:x2],
            (IMG_SIZE[1], IMG_SIZE[0]),
            interpolation=cv2.INTER_LINEAR,
        )

        if self.augment and random.random() > 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
            hard_mask = np.fliplr(hard_mask).copy()
            soft_mask = np.fliplr(soft_mask).copy()
            weight_mask = np.fliplr(weight_mask).copy()

        if self.augment and random.random() > 0.3:
            img = seg_color_jitter(img)

        return convert_seg_sample(img, hard_mask, soft_mask, weight_mask)


def seg_collate_fn(batch):
    """Custom collate function for segmentation batches (dict of tensors)."""
    return {
        "images": torch.stack([item["image"] for item in batch]),
        "hard_target": torch.stack([item["hard_target"] for item in batch]),
        "soft_target": torch.stack([item["soft_target"] for item in batch]),
        "weight": torch.stack([item["weight"] for item in batch]),
    }



In [22]:
# -- 5c. Loss function and metrics -------------------------------------------

def segmentation_loss(logits, hard_target, soft_target, weight):
    """Combined loss: weighted CE + soft BCE + weighted Dice.

    Hard targets with IGNORE_INDEX are excluded from CE loss.
    Soft BCE enables smooth supervision from averaged teacher predictions.
    Dice loss adds region-level supervision for better boundary quality.
    """
    hard_target = hard_target.long()
    soft_target = soft_target.float()
    weight = weight.float()

    valid_mask = (hard_target != IGNORE_INDEX).float()
    ce_target = hard_target.clone()
    ce_target[hard_target == IGNORE_INDEX] = 0

    ce_map = F.cross_entropy(logits, ce_target, reduction="none")
    ce_loss = (ce_map * weight * valid_mask).sum() / (weight * valid_mask).sum().clamp(min=1.0)

    road_logit = logits[:, 1] - logits[:, 0]
    bce_map = F.binary_cross_entropy_with_logits(road_logit, soft_target, reduction="none")
    bce_loss = (bce_map * weight).sum() / weight.sum().clamp(min=1.0)

    road_prob = torch.sigmoid(road_logit)
    intersection = (road_prob * soft_target * weight).sum(dim=(1, 2))
    denominator = ((road_prob + soft_target) * weight).sum(dim=(1, 2))
    dice_loss = 1.0 - ((2.0 * intersection + 1.0) / (denominator + 1.0))
    dice_loss = dice_loss.mean()

    total = ce_loss + 0.5 * bce_loss + 0.5 * dice_loss
    return total, {
        "ce": float(ce_loss.item()),
        "bce": float(bce_loss.item()),
        "dice": float(dice_loss.item()),
    }


def compute_iou(preds, targets):
    """Compute per-sample IoU for the road class."""
    ious = []
    for pred, target in zip(preds, targets):
        pred = pred.astype(bool)
        target = target.astype(bool)
        intersection = np.logical_and(pred, target).sum()
        union = np.logical_or(pred, target).sum()
        ious.append(1.0 if union == 0 else intersection / union)
    return ious


def eval_iou_on_loader(model, loader, desc):
    """Evaluate mean IoU on a DataLoader."""
    model.eval()
    ious = []
    with torch.no_grad():
        for batch in tqdm(loader, desc=desc):
            imgs = batch["images"].to(device)
            hard_target = batch["hard_target"].cpu().numpy()
            logits = model(imgs)
            preds = (logits.argmax(dim=1) == 1).cpu().numpy()
            ious.extend(compute_iou(preds, hard_target))
    return float(np.mean(ious))


In [23]:
# -- 5d. Staged optimizer and backbone mode -----------------------------------

def build_seg_optimizer(model, stage):
    """Build optimizer with staged learning rates for progressive unfreezing.

    Stage 0: Only head is trainable (warmup).
    Stage 1: Head + backbone layer4.
    Stage 2: Head + backbone layer4 + layer3.
    """
    head_params = []
    layer4_params = []
    layer3_params = []

    for name, param in model.named_parameters():
        param.requires_grad = False
        if name.startswith("backbone."):
            if stage >= 1 and ".layer4." in name:
                param.requires_grad = True
                layer4_params.append(param)
            elif stage >= 2 and ".layer3." in name:
                param.requires_grad = True
                layer3_params.append(param)
        else:
            param.requires_grad = True
            head_params.append(param)

    groups = [{"params": head_params, "lr": HEAD_LR}]
    if layer4_params:
        groups.append({"params": layer4_params, "lr": LAYER4_LR})
    if layer3_params:
        groups.append({"params": layer3_params, "lr": LAYER3_LR})

    optimizer = torch.optim.Adam(groups)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=2, factor=0.5
    )
    return optimizer, scheduler


def set_backbone_mode(model, stage):
    """Set backbone layers to train/eval mode based on which are unfrozen."""
    model.train()
    model.backbone.eval()
    if stage >= 1 and hasattr(model.backbone, "layer4"):
        model.backbone.layer4.train()
    if stage >= 2 and hasattr(model.backbone, "layer3"):
        model.backbone.layer3.train()

In [24]:
# -- 5e. Self-training round loop ---------------------------------------------

def train_segmentation_round(
    round_idx,
    start_checkpoint,
    cityscapes_dataset,
    selected_frames,
    crop_specs,
    val_loader,
):
    """Run one round of iterative self-training.

    Combines Cityscapes supervised data with pseudo-labeled hard negatives
    and their focus crops. Uses staged backbone unfreezing over epochs.
    Returns the path to the best checkpoint, training history, and best IoU.
    """
    full_dataset = HardNegativeDataset(selected_frames, augment=True)
    crop_dataset = HardCropDataset(selected_frames, crop_specs, augment=True)

    combined_dataset = ConcatDataset(
        [
            cityscapes_dataset,
            full_dataset,
            full_dataset,
            crop_dataset,
            crop_dataset,
        ]
    )

    train_loader = DataLoader(
        combined_dataset,
        batch_size=TRAIN_BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        collate_fn=seg_collate_fn,
    )

    model = load_segmentation_model(start_checkpoint)
    best_iou = 0.0
    history = []
    best_round_path = os.path.join(
        SPRINT3_OUT, f"hard_negative_retrained_round_{round_idx + 1}.pth"
    )

    stage = -1
    optimizer = None
    scheduler = None

    print(f"\nStarting self-training round {round_idx + 1} ...")
    for epoch in range(ROUND_EPOCHS):
        desired_stage = 0
        if epoch >= WARMUP_EPOCHS:
            desired_stage = 1
        if epoch >= UNFREEZE_LAYER3_EPOCH:
            desired_stage = 2

        if desired_stage != stage:
            stage = desired_stage
            optimizer, scheduler = build_seg_optimizer(model, stage)

        set_backbone_mode(model, stage)
        train_loss = 0.0
        train_steps = 0

        for batch in tqdm(train_loader, desc=f"Round {round_idx + 1} Epoch {epoch + 1}/{ROUND_EPOCHS} [train]"):
            imgs = batch["images"].to(device)
            hard_target = batch["hard_target"].to(device)
            soft_target = batch["soft_target"].to(device)
            weight = batch["weight"].to(device)

            optimizer.zero_grad()
            logits = model(imgs)
            loss, _ = segmentation_loss(logits, hard_target, soft_target, weight)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                [param for param in model.parameters() if param.requires_grad], max_norm=5.0
            )
            optimizer.step()

            train_loss += float(loss.item())
            train_steps += 1

        avg_loss = train_loss / max(train_steps, 1)
        val_iou = eval_iou_on_loader(
            model, val_loader, desc=f"Round {round_idx + 1} Epoch {epoch + 1}/{ROUND_EPOCHS} [val]"
        )
        scheduler.step(val_iou)

        current_lrs = sorted({group["lr"] for group in optimizer.param_groups}, reverse=True)
        history.append(
            {
                "round": round_idx + 1,
                "epoch": epoch + 1,
                "train_loss": avg_loss,
                "val_iou": val_iou,
                "lrs": current_lrs,
                "stage": stage,
            }
        )
        print(
            f"Round {round_idx + 1} Epoch {epoch + 1}: loss={avg_loss:.4f}, "
            f"val_IoU={val_iou:.4f}, stage={stage}, lrs={current_lrs}"
        )

        if val_iou > best_iou:
            best_iou = val_iou
            torch.save(
                {
                    "round": round_idx + 1,
                    "epoch": epoch + 1,
                    "model_state": model.state_dict(),
                    "optimizer_state": optimizer.state_dict(),
                    "val_iou": val_iou,
                },
                best_round_path,
            )
            print(f"  -> Saved new best model for round {round_idx + 1} (IoU={val_iou:.4f})")

    return best_round_path, history, best_iou, len(combined_dataset)


load_kaggle_credentials()
import kagglehub

print("Downloading Cityscapes dataset ...")
dataset_path = kagglehub.dataset_download("shuvoalok/cityscapes")
print(f"Dataset path: {dataset_path}")

train_img_dir, train_seg_dir = resolve_split_dirs(dataset_path, "train")
val_img_dir, val_seg_dir = resolve_split_dirs(dataset_path, "val")

cityscapes_dataset = CityscapesRoad(train_img_dir, train_seg_dir, augment=True)
val_full_dataset = CityscapesRoad(val_img_dir, val_seg_dir, augment=False)
val_indices = random.sample(range(len(val_full_dataset)), min(100, len(val_full_dataset)))
val_subset = Subset(val_full_dataset, val_indices)
val_loader = DataLoader(val_subset, batch_size=TRAIN_BATCH_SIZE, shuffle=False, num_workers=2, collate_fn=seg_collate_fn)
val_full_loader = DataLoader(
    val_full_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=False, num_workers=2, collate_fn=seg_collate_fn
)

round_histories = []
round_preview_paths = []
current_checkpoint = AUGMENTED_MODEL_PATH
teacher_prob_maps = [probs_city, probs_aug]
teacher_entropy_maps = [entropy_city, entropy_aug]
round_outputs = []

for round_idx in range(SELF_TRAIN_ROUNDS):
    round_label_stats = generate_pseudo_labels(
        hard_negatives, teacher_prob_maps, teacher_entropy_maps, round_idx
    )
    round_preview_paths.extend(round_label_stats["preview_paths"])

    trainable_frames = [
        meta
        for meta in hard_negatives
        if meta["pseudo_valid_ratio"] >= PSEUDO_MIN_VALID_RATIO
        and meta["pseudo_mean_weight"] >= PSEUDO_MIN_MEAN_WEIGHT
    ]
    crop_specs = build_hard_crop_specs(trainable_frames, HARD_CROPS_PER_FRAME)

    print(
        f"Round {round_idx + 1}: {len(trainable_frames)} stable hard negatives, "
        f"{len(crop_specs)} hard crops"
    )

    round_path, history, best_iou, train_size = train_segmentation_round(
        round_idx,
        current_checkpoint,
        cityscapes_dataset,
        trainable_frames,
        crop_specs,
        val_loader,
    )
    round_histories.append(
        {
            "round": round_idx + 1,
            "history": history,
            "best_iou": best_iou,
            "train_size": train_size,
            "stable_frames": len(trainable_frames),
            "hard_crops": len(crop_specs),
            "mean_valid_ratio": round_label_stats["mean_valid_ratio"],
            "mean_ignored_ratio": round_label_stats["mean_ignored_ratio"],
            "mean_weight": round_label_stats["mean_weight"],
        }
    )

    round_model = load_segmentation_model(round_path)
    round_probs, round_preds, round_entropy, _ = run_inference(
        round_model, frame_loader, desc=f"Round {round_idx + 1} teacher + TTA", use_tta=True
    )
    round_outputs.append((round_probs, round_preds, round_entropy, round_path))
    teacher_prob_maps = [probs_city, probs_aug, round_probs]
    teacher_entropy_maps = [entropy_city, entropy_aug, round_entropy]
    current_checkpoint = round_path

best_round = max(round_histories, key=lambda item: item["best_iou"])
best_round_output = max(round_outputs, key=lambda item: torch.load(item[3], map_location="cpu", weights_only=False)["val_iou"])
shutil.copyfile(best_round_output[3], RETRAINED_MODEL_PATH)
print(f"[OK] Best iterative model copied to {RETRAINED_MODEL_PATH}")

model_retrained = load_segmentation_model(RETRAINED_MODEL_PATH)
probs_retrain, preds_retrain, entropy_retrain, _ = run_inference(
    model_retrained, frame_loader, desc="Final retrained model + TTA", use_tta=True
)

100%|██████████| 199M/199M [00:05<00:00, 36.6MB/s]

Extracting files...


Dataset path: /root/.cache/kagglehub/datasets/shuvoalok/cityscapes/versions/2
Cityscapes (train): 2975 image/label pairs
Cityscapes (eval): 500 image/label pairs


Pseudo-labelling round 1: 100%|██████████| 72/72 [00:01<00:00, 49.55it/s]


Round 1: 72 stable hard negatives, 144 hard crops
Hard negatives (full frames): 72
Hard negatives (crops): 144

Starting self-training round 1 ...


...

Round 2 Epoch 10/10 [val]: 100%|██████████| 25/25 [00:01<00:00, 18.04it/s]


Round 2 Epoch 10: loss=0.1107, val_IoU=0.8329, stage=2, lrs=[0.0005, 0.0001, 5e-05]


Round 2 teacher + TTA: 100%|██████████| 475/475 [00:56<00:00,  8.36it/s]


[OK] Best iterative model copied to /content/drive/My Drive/Colab Notebooks/Computer Vision/sprint3_output/hard_negative_retrained_model.pth


Final retrained model + TTA: 100%|██████████| 475/475 [01:11<00:00,  6.66it/s]


In [30]:
"""
===============================================================================
SECTION 6: Detector Pseudo-Labelling (Grounding DINO) & Mixed Fine-Tuning
===============================================================================
Use Grounding DINO (zero-shot text-prompted detector) to generate high-quality
bounding box annotations on Ghanaian frames, then fine-tune the Sprint 2 detector
on a mix of these pseudo-labels and BDD100K source data.

Fallback: If Grounding DINO is unavailable, uses the weak detector proposals
verified by CLIP crop matching.
"""

print("\n" + "=" * 60)
print("SECTION 6: Detector Pseudo-Labelling (Grounding DINO) & Mixed Fine-Tuning")
print("=" * 60)

# -- Grounding DINO text prompts mapped to our detection classes ---------------
GDINO_PROMPTS = {
    "car": 0, "taxi": 0,
    "bus": 1,
    "truck": 2, "lorry": 2,
    "person": 3, "pedestrian": 3,
    "motorcycle rider": 4,
    "motorcycle": 5, "motorbike": 5,
    "bicycle": 6,
    "traffic sign": 7,
    "traffic light": 8,
}
# Build the prompt string for Grounding DINO (period-separated)
GDINO_PROMPT_TEXT = " . ".join(GDINO_PROMPTS.keys()) + " ."


def load_grounding_dino():
    """Load Grounding DINO SwinT model. Downloads weights on first run.
    Returns (model, predict_fn) or (None, None) if unavailable.
    """
    try:
        # Patch: newer transformers removed BertModel.get_head_mask.
        # Grounding DINO always calls it with head_mask=None so we add a
        # minimal shim that just returns [None] * num_layers.
        import transformers
        _BertModel = transformers.models.bert.modeling_bert.BertModel
        if not hasattr(_BertModel, "get_head_mask"):
            def _get_head_mask(self, head_mask, num_hidden_layers, is_attention_chunked=False):
                if head_mask is not None:
                    if head_mask.dim() == 1:
                        head_mask = head_mask.unsqueeze(0).unsqueeze(0).unsqueeze(-1).unsqueeze(-1)
                        head_mask = head_mask.expand(num_hidden_layers, -1, -1, -1, -1)
                    elif head_mask.dim() == 2:
                        head_mask = head_mask.unsqueeze(1).unsqueeze(-1).unsqueeze(-1)
                    return head_mask
                return [None] * num_hidden_layers
            _BertModel.get_head_mask = _get_head_mask
            print("[PATCH] Added BertModel.get_head_mask for Grounding DINO compatibility")

        # Patch: Grounding DINO's old BERT forward() passes `device` as the
        # 3rd arg to get_extended_attention_mask, but newer transformers
        # interprets that arg as `dtype`. Intercept and fix.
        _orig_get_extended = _BertModel.get_extended_attention_mask
        def _patched_get_extended(self, attention_mask, input_shape, dtype=None):
            if isinstance(dtype, torch.device):
                dtype = torch.float32
            return _orig_get_extended(self, attention_mask, input_shape, dtype)
        _BertModel.get_extended_attention_mask = _patched_get_extended
        print("[PATCH] Fixed get_extended_attention_mask dtype/device compat")

        from groundingdino.util.inference import load_model, predict
        import groundingdino
    except ImportError:
        print("WARNING: groundingdino not available, falling back to weak detector pseudo-labels.")
        return None, None

    # Download Grounding DINO weights if not present
    gdino_config = os.path.join(
        os.path.dirname(groundingdino.__file__),
        "config", "GroundingDINO_SwinT_OGC.py"
    )
    gdino_weights = os.path.join(SPRINT3_OUT, "groundingdino_swint_ogc.pth")
    if not os.path.isfile(gdino_weights):
        print("Downloading Grounding DINO SwinT weights ...")
        import urllib.request
        url = "https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth"
        urllib.request.urlretrieve(url, gdino_weights)
        print(f"[OK] Downloaded to {gdino_weights}")

    model = load_model(gdino_config, gdino_weights, device=str(device))
    return model, predict


def map_gdino_label_to_class(phrase):
    """Map a Grounding DINO detected phrase to our 9-class detection index."""
    phrase = phrase.lower().strip()
    if phrase in GDINO_PROMPTS:
        return GDINO_PROMPTS[phrase]
    # Fuzzy match: check if any key is a substring
    for key, cls_id in GDINO_PROMPTS.items():
        if key in phrase or phrase in key:
            return cls_id
    return -1


def generate_detection_pseudo_labels_gdino(gdino_model, gdino_predict):
    """Generate pseudo bbox labels using Grounding DINO as strong teacher.

    For each hard-negative frame, runs Grounding DINO with text prompts for
    all 9 object classes, maps detected phrases to class indices, and applies
    per-class NMS to remove duplicate boxes.
    """
    from groundingdino.util.inference import load_image
    import torchvision.ops as ops

    pseudo_labels = []
    for meta in tqdm(hard_negatives, desc="Grounding DINO pseudo-labels"):
        # Grounding DINO expects its own image loading
        image_source, image_tensor = load_image(meta["path"])
        src_h, src_w = image_source.shape[:2]

        boxes_norm, logits, phrases = gdino_predict(
            model=gdino_model,
            image=image_tensor,
            caption=GDINO_PROMPT_TEXT,
            box_threshold=GDINO_BOX_THRESH,
            text_threshold=GDINO_TEXT_THRESH,
        )

        if len(boxes_norm) == 0:
            continue

        # Convert cxcywh normalized -> x1y1x2y2 in detector pixel coords
        kept_boxes = []
        kept_labels = []
        kept_scores = []

        for box_norm, logit, phrase in zip(boxes_norm, logits, phrases):
            cls_id = map_gdino_label_to_class(phrase)
            if cls_id < 0:
                continue

            cx, cy, w, h = box_norm.tolist()
            x1 = (cx - w / 2) * DET_IMG_W
            y1 = (cy - h / 2) * DET_IMG_H
            x2 = (cx + w / 2) * DET_IMG_W
            y2 = (cy + h / 2) * DET_IMG_H

            # Clamp to image bounds
            x1 = max(0, min(DET_IMG_W, x1))
            y1 = max(0, min(DET_IMG_H, y1))
            x2 = max(0, min(DET_IMG_W, x2))
            y2 = max(0, min(DET_IMG_H, y2))

            if (x2 - x1) < DET_MIN_BOX_EDGE or (y2 - y1) < DET_MIN_BOX_EDGE:
                continue

            kept_boxes.append(torch.tensor([x1, y1, x2, y2], dtype=torch.float32))
            kept_labels.append(cls_id)
            kept_scores.append(float(logit.item()))

        if not kept_boxes:
            continue

        # NMS per class to remove duplicates
        all_boxes = torch.stack(kept_boxes)
        all_scores = torch.tensor(kept_scores, dtype=torch.float32)
        all_labels = torch.tensor(kept_labels, dtype=torch.long)

        final_boxes = []
        final_labels = []
        final_scores = []
        for cls_id in all_labels.unique().tolist():
            cls_mask = all_labels == cls_id
            cls_boxes = all_boxes[cls_mask]
            cls_scores = all_scores[cls_mask]
            keep = ops.nms(cls_boxes, cls_scores, GDINO_NMS_THRESH)
            final_boxes.append(cls_boxes[keep])
            final_labels.append(all_labels[cls_mask][keep])
            final_scores.append(cls_scores[keep])

        pseudo_labels.append(
            {
                "path": meta["path"],
                "boxes": torch.cat(final_boxes, dim=0),
                "labels": torch.cat(final_labels, dim=0),
                "scores": torch.cat(final_scores, dim=0),
            }
        )

    return pseudo_labels



SECTION 6: Detector Pseudo-Labelling (Grounding DINO) & Mixed Fine-Tuning


In [31]:
# -- 6b. Fallback: weak detector + CLIP verification -------------------------

def generate_detection_pseudo_labels_weak():
    """Fallback pseudo-labeller: uses the Sprint 2 detector's own proposals,
    filtered by BLIP/CLIP class hints and verified by CLIP crop matching.
    Circular but still better than no labels when GDINO is unavailable.
    """
    clip_model_det, _, clip_preprocess_det = open_clip.create_model_and_transforms(
        "ViT-B-32", pretrained="laion2b_s34b_b79k"
    )
    clip_model_det = clip_model_det.to(device).eval()
    tokenizer_det = open_clip.get_tokenizer("ViT-B-32")

    class_features = []
    with torch.no_grad():
        for class_name in DETECT_CLASSES:
            prompts = [f"a dashboard camera photo of {p}" for p in OBJECT_PROMPTS[class_name]]
            tokens = tokenizer_det(prompts).to(device)
            feats = clip_model_det.encode_text(tokens)
            feats = feats / feats.norm(dim=-1, keepdim=True)
            class_feat = feats.mean(dim=0)
            class_feat = class_feat / class_feat.norm()
            class_features.append(class_feat)
    class_text_features = torch.stack(class_features, dim=0)

    CAPTION_TO_CLASS = {
        "car": 0, "cars": 0, "taxi": 0, "vehicle": 0, "vehicles": 0,
        "bus": 1, "buses": 1,
        "truck": 2, "trucks": 2, "lorry": 2,
        "pedestrian": 3, "pedestrians": 3, "person": 3, "people": 3,
        "man": 3, "woman": 3, "child": 3,
        "rider": 4, "riders": 4,
        "motorcycle": 5, "motorcycles": 5, "okada": 5, "motorbike": 5,
        "bicycle": 6, "bicycles": 6, "bike": 6, "cyclist": 6,
        "sign": 7, "signs": 7,
        "traffic light": 8, "traffic lights": 8, "signal": 8,
    }

    pseudo_labels = []
    with torch.no_grad():
        for meta in tqdm(hard_negatives, desc="Weak-detector pseudo-labels"):
            caption = f"{meta.get('blip_caption', '')} {meta.get('blip_road_caption', '')}"
            class_hints = set()
            for kw, cid in CAPTION_TO_CLASS.items():
                if re.search(rf"\b{re.escape(kw)}\b", caption.lower()):
                    class_hints.add(cid)
            for cn in meta.get("clip_likely_classes", []):
                if cn in DETECT_CLASSES:
                    class_hints.add(DETECT_CLASSES.index(cn))
            if not class_hints:
                continue

            img_pil = Image.open(meta["path"]).convert("RGB")
            det_img = resize_for_detection(img_pil)
            det_tensor = image_to_det_tensor(det_img).unsqueeze(0).to(device)
            results = model_detector(det_tensor)[0]
            boxes, scores, labels = results["boxes"], results["scores"], results["labels"]
            if boxes.shape[0] == 0:
                continue

            boxes, scores, labels = road_aware_filter(
                boxes, scores, labels,
                torch.from_numpy(preds_retrain[meta["idx"]].astype(np.uint8)),
                DET_IMG_H, DET_IMG_W,
            )

            kept_boxes, kept_labels, kept_scores = [], [], []
            for box, score, label in zip(boxes, scores, labels):
                lid = int(label.item())
                if lid not in class_hints or float(score.item()) < DET_PSEUDO_SCORE_THRESH:
                    continue
                # CLIP verification
                orig_w, orig_h = img_pil.size
                sx, sy = orig_w / DET_IMG_W, orig_h / DET_IMG_H
                bx1, by1, bx2, by2 = [int(round(v)) for v in box.cpu().numpy().tolist()]
                bx1, by1 = int(max(0, bx1 * sx)), int(max(0, by1 * sy))
                bx2, by2 = int(min(orig_w, bx2 * sx)), int(min(orig_h, by2 * sy))
                if bx2 - bx1 < DET_MIN_BOX_EDGE or by2 - by1 < DET_MIN_BOX_EDGE:
                    continue
                crop = img_pil.crop((bx1, by1, bx2, by2))
                ct = clip_preprocess_det(crop).unsqueeze(0).to(device)
                feat = clip_model_det.encode_image(ct)
                feat = feat / feat.norm(dim=-1, keepdim=True)
                sims = (feat @ class_text_features.T).squeeze(0)
                best_idx = int(torch.argmax(sims).item())
                if best_idx != lid or float(sims[best_idx]) < DET_CLIP_VERIFY_THRESH:
                    continue
                kept_boxes.append(box.cpu())
                kept_labels.append(label.cpu())
                kept_scores.append(max(float(score.item()), float(sims[best_idx])))

            if kept_boxes:
                pseudo_labels.append({
                    "path": meta["path"],
                    "boxes": torch.stack(kept_boxes, dim=0),
                    "labels": torch.stack(kept_labels, dim=0),
                    "scores": torch.tensor(kept_scores, dtype=torch.float32),
                })

    del clip_model_det, class_text_features
    torch.cuda.empty_cache()
    return pseudo_labels


# Try Grounding DINO first, fall back to weak detector + CLIP
gdino_model, gdino_predict = load_grounding_dino()
if gdino_model is not None:
    print("Using Grounding DINO as strong teacher for detection pseudo-labels ...")
    det_pseudo_labels = generate_detection_pseudo_labels_gdino(gdino_model, gdino_predict)
    del gdino_model
    torch.cuda.empty_cache()
else:
    print("Falling back to weak detector + CLIP verification ...")
    det_pseudo_labels = generate_detection_pseudo_labels_weak()

print(f"[OK] Generated pseudo-labels for {len(det_pseudo_labels)} Ghana frames")
if det_pseudo_labels:
    total_boxes = sum(item["boxes"].shape[0] for item in det_pseudo_labels)
    print(f"  Total pseudo boxes: {total_boxes}")
    pseudo_class_counts = defaultdict(int)
    for item in det_pseudo_labels:
        for label in item["labels"].tolist():
            pseudo_class_counts[DETECT_CLASSES[label]] += 1
    for class_name, count in sorted(pseudo_class_counts.items(), key=lambda pair: (-pair[1], pair[0])):
        print(f"  {class_name}: {count}")

[PATCH] Fixed get_extended_attention_mask dtype/device compat
final text_encoder_type: bert-base-uncased


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Using Grounding DINO as strong teacher for detection pseudo-labels ...


Grounding DINO pseudo-labels:   0%|          | 0/72 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/usr/local/lib/python3.12/dist-packages/groundingdino/models/GroundingDINO/transformer.py:862: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=F

[OK] Generated pseudo-labels for 64 Ghana frames
  Total pseudo boxes: 315
  car: 80
  truck: 73
  pedestrian: 66
  rider: 33
  bicycle: 17
  motorcycle: 16
  traffic sign: 16
  bus: 11
  traffic light: 3


In [32]:
# -- 6c. Detection pseudo-label datasets --------------------------------------

class DetectionPseudoDataset(Dataset):
    """Dataset of Ghanaian frames with pseudo bounding box annotations."""
    def __init__(self, labels):
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = self.labels[idx]
        img = Image.open(item["path"]).convert("RGB")
        img = resize_for_detection(img)
        if random.random() > 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
            boxes = item["boxes"].clone()
            x1 = boxes[:, 0].clone()
            x2 = boxes[:, 2].clone()
            boxes[:, 0] = DET_IMG_W - x2
            boxes[:, 2] = DET_IMG_W - x1
        else:
            boxes = item["boxes"].clone()

        if random.random() > 0.3:
            img = det_color_jitter(img)

        return image_to_det_tensor(img), {"boxes": boxes.float(), "labels": item["labels"].long()}


class BDDRoadDetectionDataset(Dataset):
    """BDD100K source detection dataset for mixed-domain training."""
    def __init__(self, image_dir, label_json, max_items=None):
        with open(label_json, "r") as handle:
            raw = json.load(handle)

        self.samples = []
        for item in raw:
            image_path = os.path.join(image_dir, item["name"])
            if not os.path.isfile(image_path):
                continue
            boxes = []
            labels = []
            for ann in item.get("labels", []):
                category = ann.get("category")
                box = ann.get("box2d")
                if category not in BDD100K_CLASS_MAP or box is None:
                    continue
                x1 = float(box["x1"])
                y1 = float(box["y1"])
                x2 = float(box["x2"])
                y2 = float(box["y2"])
                if x2 <= x1 or y2 <= y1:
                    continue
                boxes.append([x1, y1, x2, y2])
                labels.append(BDD100K_CLASS_MAP[category])
            if boxes:
                self.samples.append({"path": image_path, "boxes": boxes, "labels": labels})

        random.shuffle(self.samples)
        if max_items is not None:
            self.samples = self.samples[:max_items]
        print(f"BDD100K source detection samples: {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        img = Image.open(sample["path"]).convert("RGB")
        orig_w, orig_h = img.size
        img = resize_for_detection(img)
        if random.random() > 0.3:
            img = det_color_jitter(img)

        boxes = torch.tensor(sample["boxes"], dtype=torch.float32)
        boxes[:, 0::2] *= DET_IMG_W / orig_w
        boxes[:, 1::2] *= DET_IMG_H / orig_h

        return image_to_det_tensor(img), {
            "boxes": boxes,
            "labels": torch.tensor(sample["labels"], dtype=torch.long),
        }


def det_collate_fn(batch):
    """Detection collate: stack images, keep targets as list of dicts."""
    images = torch.stack([item[0] for item in batch])
    targets = [item[1] for item in batch]
    return images, targets


SOURCE_DET_ROOT = os.environ.get("BDD100K_ROOT", os.path.join(SHARED_CV, "bdd100k"))
SOURCE_DET_IMAGES = os.environ.get(
    "BDD100K_IMAGES", os.path.join(SOURCE_DET_ROOT, "images", "100k", "train")
)
SOURCE_DET_LABELS = os.environ.get(
    "BDD100K_LABELS", os.path.join(SOURCE_DET_ROOT, "labels", "det_20", "det_train.json")
)

source_det_dataset = None
if os.path.isdir(SOURCE_DET_IMAGES) and os.path.isfile(SOURCE_DET_LABELS):
    source_det_dataset = BDDRoadDetectionDataset(
        SOURCE_DET_IMAGES, SOURCE_DET_LABELS, max_items=MAX_SOURCE_DET_IMAGES
    )
else:
    print("BDD100K source detection data not found; detector fine-tuning will use target pseudo-labels only.")

BDD100K source detection data not found; detector fine-tuning will use target pseudo-labels only.


In [33]:
# -- 6d. Fine-tune detection model on pseudo-labels + source data -------------

def train_detection_model():
    """Fine-tune the Sprint 2 detector on Grounding DINO pseudo-labels (Ghana)
    + BDD100K source data (if available). Backbone is frozen; only FPN + head are trained.
    """
    if len(det_pseudo_labels) < 10:
        print("Not enough verified pseudo-labeled Ghana frames for detector retraining.")
        return model_detector, []

    datasets = []
    if source_det_dataset is not None and len(source_det_dataset) > 0:
        datasets.append(source_det_dataset)
    pseudo_dataset = DetectionPseudoDataset(det_pseudo_labels)
    datasets.append(ConcatDataset([pseudo_dataset] * DET_PSEUDO_OVERSAMPLE))

    train_dataset = datasets[0] if len(datasets) == 1 else ConcatDataset(datasets)
    train_loader = DataLoader(
        train_dataset,
        batch_size=DET_BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        collate_fn=det_collate_fn,
    )

    model = RoadAwareDetector(num_classes=NUM_DETECT_CLASSES, pretrained_backbone=False)
    det_ckpt = torch.load(DETECTOR_MODEL_PATH, map_location=device, weights_only=False)
    model.load_state_dict(det_ckpt["model_state"])
    model.to(device)

    for layer in [model.stem, model.layer1, model.layer2, model.layer3, model.layer4]:
        for param in layer.parameters():
            param.requires_grad = False

    trainable = [param for param in model.parameters() if param.requires_grad]
    optimizer = torch.optim.Adam(trainable, lr=1e-4)
    history = []

    for epoch in range(DET_EPOCHS):
        model.train()
        model.stem.eval()
        model.layer1.eval()
        model.layer2.eval()
        model.layer3.eval()
        model.layer4.eval()

        epoch_loss = 0.0
        n_batches = 0
        for imgs, targets in tqdm(train_loader, desc=f"Det Epoch {epoch + 1}/{DET_EPOCHS}"):
            imgs = imgs.to(device)
            targets = [{key: value.to(device) for key, value in target.items()} for target in targets]

            optimizer.zero_grad()
            losses = model(imgs, targets)
            total_loss = losses["total_loss"]
            if torch.isfinite(total_loss) and total_loss.item() > 0:
                total_loss.backward()
                torch.nn.utils.clip_grad_norm_(trainable, max_norm=5.0)
                optimizer.step()

            epoch_loss += float(total_loss.item())
            n_batches += 1

        avg_loss = epoch_loss / max(n_batches, 1)
        history.append({"epoch": epoch + 1, "loss": avg_loss})
        print(f"  Det Epoch {epoch + 1}: loss={avg_loss:.4f}")

    torch.save(
        {
            "epoch": DET_EPOCHS,
            "model_state": model.state_dict(),
            "img_h": DET_IMG_H,
            "img_w": DET_IMG_W,
            "pseudo_labels_count": len(det_pseudo_labels),
            "source_labels_count": len(source_det_dataset) if source_det_dataset is not None else 0,
        },
        RETRAINED_DETECTOR_PATH,
    )
    print(f"[OK] Retrained detector saved to {RETRAINED_DETECTOR_PATH}")
    model.eval()
    return model, history


model_det_retrain, det_training_history = train_detection_model()

Det Epoch 1/6: 100%|██████████| 64/64 [00:03<00:00, 20.96it/s]


  Det Epoch 1: loss=1.1685


Det Epoch 2/6: 100%|██████████| 64/64 [00:03<00:00, 21.05it/s]


  Det Epoch 2: loss=0.6811


Det Epoch 3/6: 100%|██████████| 64/64 [00:02<00:00, 21.59it/s]


...

  Det Epoch 4: loss=0.3733


Det Epoch 5/6: 100%|██████████| 64/64 [00:03<00:00, 21.33it/s]


  Det Epoch 5: loss=0.2680


Det Epoch 6/6: 100%|██████████| 64/64 [00:03<00:00, 21.01it/s]


  Det Epoch 6: loss=0.2011
[OK] Retrained detector saved to /content/drive/My Drive/Colab Notebooks/Computer Vision/sprint3_output/retrained_detector.pth


In [34]:
"""
===============================================================================
SECTION 7: Before/After Evaluation
===============================================================================
Compare Cityscapes-only, Ghana-augmented, and hard-negative-retrained models
on Ghanaian frames. Also compare detection with different road filters and
original vs retrained detector.
"""

print("\n" + "=" * 60)
print("SECTION 7: Before/After Evaluation")
print("=" * 60)


# -- 7a. Segmentation model comparison on Ghanaian frames ---------------------

def model_summary(name, probs, preds, entropy):
    """Compute summary metrics for a model's predictions on Ghanaian frames."""
    uncertain = ((probs > 0.3) & (probs < 0.7)).mean()
    return {
        "model": name,
        "mean_road_prob": float(probs.mean()),
        "mean_entropy": float(entropy.mean()),
        "road_ratio": float(preds.mean()),
        "uncertain_pixel_ratio": float(uncertain),
    }


summaries = [
    model_summary("Cityscapes-only", probs_city, preds_city, entropy_city),
    model_summary("Ghana-augmented", probs_aug, preds_aug, entropy_aug),
    model_summary("Hard-neg retrained", probs_retrain, preds_retrain, entropy_retrain),
]

print(f"\n{'Model':<25} {'MeanRoadProb':>13} {'MeanEntropy':>13} {'RoadRatio':>11} {'UncertainPx':>13}")
print("-" * 78)
for summary in summaries:
    print(
        f"{summary['model']:<25} {summary['mean_road_prob']:>13.4f} "
        f"{summary['mean_entropy']:>13.4f} {summary['road_ratio']:>11.4f} "
        f"{summary['uncertain_pixel_ratio']:>13.4f}"
    )


SECTION 7: Before/After Evaluation

Model                      MeanRoadProb   MeanEntropy   RoadRatio   UncertainPx
------------------------------------------------------------------------------
Cityscapes-only                  0.1510        0.1112      0.1540        0.0735
Ghana-augmented                  0.2197        0.0956      0.2224        0.0507
Hard-neg retrained               0.2100        0.0526      0.2114        0.0196


In [35]:
# -- 7b. Cityscapes val IoU regression check ----------------------------------

iou_city = eval_iou_on_loader(model_cityscape, val_full_loader, "Cityscapes model val")
iou_aug = eval_iou_on_loader(model_augmented, val_full_loader, "Augmented model val")
iou_retrain = eval_iou_on_loader(model_retrained, val_full_loader, "Retrained model val")

print(f"\n{'Model':<25} {'Cityscapes Val IoU':>20}")
print("-" * 47)
print(f"{'Cityscapes-only':<25} {iou_city:>20.4f}")
print(f"{'Ghana-augmented':<25} {iou_aug:>20.4f}")
print(f"{'Hard-neg retrained':<25} {iou_retrain:>20.4f}")

assert iou_retrain >= 0.75, f"REGRESSION: Retrained model IoU {iou_retrain:.4f} dropped below 0.75!"
print("[OK] No significant regression on Cityscapes validation")

Retrained model val: 100%|██████████| 125/125 [00:04<00:00, 27.32it/s]


Model                       Cityscapes Val IoU
-----------------------------------------------
Cityscapes-only                         0.8446
Ghana-augmented                         0.8198
Hard-neg retrained                      0.8612
[OK] No significant regression on Cityscapes validation


In [36]:
# -- 7c. Detection comparison: road filter + detector impact ------------------

def run_detection_with_filter(det_model, eval_frames, road_masks, desc, threshold=0.1):
    """Run a detector on frames with road-aware filtering from a given seg model.
    Returns total detections, mean score, per-class counts, and dets/frame.
    """
    total_dets = 0
    total_scores = []
    per_class_totals = {class_name: 0 for class_name in DETECT_CLASSES}

    with torch.no_grad():
        for meta in tqdm(eval_frames, desc=desc):
            img_pil = Image.open(meta["path"]).convert("RGB")
            det_img = resize_for_detection(img_pil)
            det_tensor = image_to_det_tensor(det_img).unsqueeze(0).to(device)
            results = det_model(det_tensor)[0]
            boxes, scores, labels = results["boxes"], results["scores"], results["labels"]

            if boxes.shape[0] > 0:
                boxes, scores, labels = road_aware_filter(
                    boxes,
                    scores,
                    labels,
                    torch.from_numpy(road_masks[meta["idx"]].astype(np.uint8)),
                    DET_IMG_H,
                    DET_IMG_W,
                )

            keep = scores > threshold
            boxes, scores, labels = boxes[keep], scores[keep], labels[keep]

            total_dets += len(boxes)
            if len(scores) > 0:
                total_scores.extend(scores.cpu().tolist())
            for class_idx in range(NUM_DETECT_CLASSES):
                per_class_totals[DETECT_CLASSES[class_idx]] += int((labels == class_idx).sum())

    return {
        "total_dets": total_dets,
        "mean_score": float(np.mean(total_scores)) if total_scores else 0.0,
        "per_class": per_class_totals,
        "dets_per_frame": total_dets / len(eval_frames),
    }


det_eval_frames = sorted(frame_metadata, key=lambda item: item["failure_score"], reverse=True)
det_with_cityscape = run_detection_with_filter(
    model_detector, det_eval_frames, preds_city, "OrigDet + Cityscapes seg"
)
det_with_augmented = run_detection_with_filter(
    model_detector, det_eval_frames, preds_aug, "OrigDet + Augmented seg"
)
det_with_retrained = run_detection_with_filter(
    model_detector, det_eval_frames, preds_retrain, "OrigDet + Retrained seg"
)
det_retrained_model = run_detection_with_filter(
    model_det_retrain, det_eval_frames, preds_retrain, "RetrainedDet + Retrained seg"
)

print(f"\n{'Configuration':<35} {'TotalDets':>10} {'Dets/Frame':>11} {'MeanScore':>10}")
print("-" * 68)
for name, result in [
    ("Orig det + Cityscapes seg", det_with_cityscape),
    ("Orig det + Augmented seg", det_with_augmented),
    ("Orig det + Retrained seg", det_with_retrained),
    ("Retrained det + Retrained seg", det_retrained_model),
]:
    print(
        f"{name:<35} {result['total_dets']:>10} "
        f"{result['dets_per_frame']:>11.1f} {result['mean_score']:>10.4f}"
    )

RetrainedDet + Retrained seg: 100%|██████████| 3797/3797 [03:55<00:00, 16.11it/s]


Configuration                        TotalDets  Dets/Frame  MeanScore
--------------------------------------------------------------------
Orig det + Cityscapes seg                 3804         1.0     0.4039
Orig det + Augmented seg                  5341         1.4     0.4166
Orig det + Retrained seg                  3842         1.0     0.4066
Retrained det + Retrained seg            12254         3.2     0.3518


In [37]:
# -- 7d. Visualization: representative before/after cases --------------------

print("\nGenerating before/after visualizations ...")
vis_frames = []
vis_counts = defaultdict(int)
for meta in det_eval_frames:
    category = meta["top_failure_category"]
    if vis_counts[category] < 2:
        vis_frames.append(meta)
        vis_counts[category] += 1
    if len(vis_frames) >= 10:
        break

for meta in vis_frames:
    idx = meta["idx"]
    orig_img = resize_for_segmentation(Image.open(meta["path"]).convert("RGB"))
    orig_np = np.array(orig_img)
    panels = [orig_np]

    for pred in [preds_city[idx], preds_aug[idx], preds_retrain[idx]]:
        overlay = orig_np.copy().astype(np.float32)
        mask = pred.astype(bool)
        overlay[mask, 1] = np.clip(overlay[mask, 1] + 100, 0, 255)
        overlay[mask, 0] = overlay[mask, 0] * 0.6
        overlay[mask, 2] = overlay[mask, 2] * 0.6
        panels.append(overlay.astype(np.uint8))

    combined = np.concatenate(panels, axis=1)
    fname = os.path.basename(meta["path"])
    Image.fromarray(combined).save(
        os.path.join(VIS_DIR, f"compare_{meta['top_failure_category']}_{fname}")
    )

print(f"[OK] Saved {len(vis_frames)} comparison images to {VIS_DIR}")


Generating before/after visualizations ...
[OK] Saved 10 comparison images to /content/drive/My Drive/Colab Notebooks/Computer Vision/sprint3_output/visualizations


In [38]:

"""
===============================================================================
SECTION 8: Failure Analysis Report
===============================================================================
Generate a comprehensive markdown report summarizing failure patterns, pseudo-label
quality, segmentation improvements, and detection results.
"""

print("\n" + "=" * 60)
print("SECTION 8: Failure Analysis Report")
print("=" * 60)

category_improvements = {}
for category in category_names:
    cat_indices = [meta["idx"] for meta in hard_negatives if meta["top_failure_category"] == category]
    if not cat_indices:
        continue
    category_improvements[category] = {
        "city_entropy": float(entropy_city[cat_indices].mean()),
        "aug_entropy": float(entropy_aug[cat_indices].mean()),
        "retrain_entropy": float(entropy_retrain[cat_indices].mean()),
        "improvement": float(entropy_aug[cat_indices].mean() - entropy_retrain[cat_indices].mean()),
    }

report_lines = []
report_lines.append("# Failure Analysis Report -- Sprint 3 (Improved)\n")
report_lines.append("## 1. Mining Strategy\n")
report_lines.append("- Balanced hard-negative selection across CLIP failure categories")
report_lines.append("- Diversity-aware selection using CLIP image embeddings")
report_lines.append("- Confidence-masked pseudo-labels with ignored pixels")
report_lines.append("- Iterative self-training with staged backbone unfreezing")
report_lines.append("- Hard-crop training around uncertainty, disagreement, and boundary regions")
report_lines.append("- Detector pseudo-label verification with CLIP crop matching\n")

report_lines.append("## 2. Hard Negative Distribution\n")
report_lines.append("| Category | Count | Percentage |")
report_lines.append("|----------|-------|------------|")
for category, count in sorted(hn_cats.items(), key=lambda pair: (-pair[1], pair[0])):
    report_lines.append(f"| {category} | {count} | {100.0 * count / len(hard_negatives):.1f}% |")

report_lines.append("")
report_lines.append("## 3. Pseudo-Label Quality\n")
report_lines.append("| Round | Stable Frames | Mean Valid Px | Mean Ignored Px | Mean Weight |")
report_lines.append("|-------|---------------|---------------|-----------------|-------------|")
for round_info in round_histories:
    report_lines.append(
        f"| {round_info['round']} | {round_info['stable_frames']} | "
        f"{round_info['mean_valid_ratio']:.4f} | {round_info['mean_ignored_ratio']:.4f} | "
        f"{round_info['mean_weight']:.4f} |"
    )

report_lines.append("")
report_lines.append("## 4. Segmentation Results\n")
report_lines.append("| Model | Cityscapes Val IoU | Ghana Mean Entropy | Ghana Uncertain Px |")
report_lines.append("|-------|-------------------|-------------------|-------------------|")
for summary in summaries:
    iou_value = {
        "Cityscapes-only": iou_city,
        "Ghana-augmented": iou_aug,
        "Hard-neg retrained": iou_retrain,
    }[summary["model"]]
    report_lines.append(
        f"| {summary['model']} | {iou_value:.4f} | {summary['mean_entropy']:.4f} | "
        f"{summary['uncertain_pixel_ratio']:.4f} |"
    )

report_lines.append("")
report_lines.append("### Per-Category Entropy Improvement\n")
report_lines.append("| Category | Before (Augmented) | After (Retrained) | Change |")
report_lines.append("|----------|-------------------|-------------------|--------|")
for category, stats in sorted(category_improvements.items(), key=lambda pair: -pair[1]["improvement"]):
    report_lines.append(
        f"| {category} | {stats['aug_entropy']:.4f} | {stats['retrain_entropy']:.4f} | "
        f"{stats['improvement']:.4f} |"
    )

report_lines.append("")
report_lines.append("### Round-by-Round Training\n")
report_lines.append("| Round | Epoch | Train Loss | Val IoU | Stage | LRs |")
report_lines.append("|-------|-------|------------|---------|-------|-----|")
for round_info in round_histories:
    for row in round_info["history"]:
        report_lines.append(
            f"| {row['round']} | {row['epoch']} | {row['train_loss']:.4f} | "
            f"{row['val_iou']:.4f} | {row['stage']} | {row['lrs']} |"
        )

report_lines.append("")
report_lines.append("## 5. Detection Results\n")
report_lines.append("| Configuration | Total Dets | Dets/Frame | Mean Score |")
report_lines.append("|---------------|-----------|------------|------------|")
for name, result in [
    ("Orig det + Cityscapes seg", det_with_cityscape),
    ("Orig det + Augmented seg", det_with_augmented),
    ("Orig det + Retrained seg", det_with_retrained),
    ("Retrained det + Retrained seg", det_retrained_model),
]:
    report_lines.append(
        f"| {name} | {result['total_dets']} | {result['dets_per_frame']:.1f} | {result['mean_score']:.4f} |"
    )

report_lines.append("")
report_lines.append(f"- Verified pseudo-labeled Ghana frames: {len(det_pseudo_labels)}")
report_lines.append(
    f"- Source detection frames used: {len(source_det_dataset) if source_det_dataset is not None else 0}"
)

if det_training_history:
    report_lines.append("")
    report_lines.append("| Det Epoch | Loss |")
    report_lines.append("|-----------|------|")
    for row in det_training_history:
        report_lines.append(f"| {row['epoch']} | {row['loss']:.4f} |")

report_lines.append("")
report_lines.append("## 6. Visualizations\n")
report_lines.append(f"- Comparison images saved to `{VIS_DIR}`")
report_lines.append(f"- Pseudo-label previews saved to `{PSEUDO_LABELS_DIR}`")

report_path = os.path.join(REPORT_DIR, "failure_analysis_report.md")
with open(report_path, "w") as handle:
    handle.write("\n".join(report_lines))

print(f"[OK] Failure Analysis Report saved to {report_path}")


catalogue = []
skip_keys = {"pseudo_hard", "pseudo_soft", "pseudo_weight", "focus_map", "selection_score"}
for meta in frame_metadata:
    entry = {}
    for key, value in meta.items():
        if key in skip_keys:
            continue
        if isinstance(value, dict):
            entry[key] = {sub_key: serialize_value(sub_val) for sub_key, sub_val in value.items()}
        elif isinstance(value, list):
            entry[key] = [serialize_value(item) for item in value]
        else:
            entry[key] = serialize_value(value)
    catalogue.append(entry)

with open(CATALOGUE_PATH, "w") as handle:
    json.dump(catalogue, handle, indent=2)

print(f"[OK] Catalogue saved to {CATALOGUE_PATH}")

print("\n" + "=" * 60)
print("SPRINT 3 IMPROVED COMPLETE")
print("=" * 60)
print(f"  Hard negatives mined:       {len(hard_negatives)}")
print(f"  Failure categories found:   {len(hn_cats)}")
print(f"  Retrained model IoU:        {best_round['best_iou']:.4f}")
print(f"  Final Cityscapes val IoU:   {iou_retrain:.4f}")
print(
    f"  Orig det (retrained seg):   {det_with_retrained['total_dets']} dets, "
    f"{det_with_retrained['dets_per_frame']:.1f}/frame"
)
print(
    f"  Retrained det:              {det_retrained_model['total_dets']} dets, "
    f"{det_retrained_model['dets_per_frame']:.1f}/frame"
)
print(f"  Detection pseudo-labels:    {len(det_pseudo_labels)} frames")
print(f"  Failure report:             {report_path}")
print(f"  Visualizations:             {VIS_DIR}")
print(f"  Failure catalogue:          {CATALOGUE_PATH}")
print("=" * 60)



SECTION 8: Failure Analysis Report
[OK] Failure Analysis Report saved to /content/drive/My Drive/Colab Notebooks/Computer Vision/sprint3_output/failure_report/failure_analysis_report.md
[OK] Catalogue saved to /content/drive/My Drive/Colab Notebooks/Computer Vision/sprint3_output/failure_catalogue.json

SPRINT 3 IMPROVED COMPLETE
  Hard negatives mined:       72
  Failure categories found:   7
  Retrained model IoU:        0.8573
  Final Cityscapes val IoU:   0.8612
  Orig det (retrained seg):   3842 dets, 1.0/frame
  Retrained det:              12254 dets, 3.2/frame
  Detection pseudo-labels:    64 frames
  Failure report:             /content/drive/My Drive/Colab Notebooks/Computer Vision/sprint3_output/failure_report/failure_analysis_report.md
  Visualizations:             /content/drive/My Drive/Colab Notebooks/Computer Vision/sprint3_output/visualizations
  Failure catalogue:          /content/drive/My Drive/Colab Notebooks/Computer Vision/sprint3_output/failure_catalogue.json
